# **Importamos las dependencias**

In [3]:
import pandas as pd

# **Cargamos los datos**

In [4]:
rutaMB = '/content/drive/MyDrive/Apuntes/Octavo semestre/Minería de datos/Programas/Práctica 1/Datos /afluenciamb_simple_04_2025.csv'
rutaRTP = '/content/drive/MyDrive/Apuntes/Octavo semestre/Minería de datos/Programas/Práctica 1/Datos /afluenciartp_desglosado_04_2025.csv'
rutaMetro = '/content/drive/MyDrive/Apuntes/Octavo semestre/Minería de datos/Programas/Práctica 1/Datos /data-2025-05-28.csv'
rutaCable = '/content/drive/MyDrive/Apuntes/Octavo semestre/Minería de datos/Programas/Práctica 1/Datos /CableBus.csv'

In [5]:
# Leemos el primer archivo
MB = pd.read_csv(rutaMB)
MB.head(1)

,fecha,anio,mes,linea,afluencia
0,2005-07-26,2005,Julio,Línea 1,3032667.0


In [6]:
RTP = pd.read_csv(rutaRTP, encoding='latin-1')
RTP.head(1)

,fecha,mes,anio,servicio,tipo_pago,afluencia
0,2022-01-01,Enero,2022.0,Servicios Temporales,Boleto,0.0


In [7]:
Metro = pd.read_csv(rutaMetro)
Metro.head(1)

,fecha,anio,mes,linea,estacion,afluencia,temporal_fecha,..anio_fecha
0,2010-01-01,2010,Enero,Linea 1,Zaragoza,20227,2010-01,2010


In [37]:
CableBus = pd.read_csv(rutaCable)
CableBus.head(1)

,fecha,mes,anio,linea,tipo_pago,afluencia,temporal_fecha,..anio_fecha
0,2022-01-01,Enero,2022,Línea 1,Prepago,24975,2022-01,2022


# **Usamos concat**

### **Limpieza de datos**

In [34]:
# --- Eliminamos las columnas que nos impiden unir los df ---

# Para el metro
columnasM = ['..anio_fecha', 'temporal_fecha', 'estacion', 'linea']
Metro = Metro.drop(columnasM, axis=1)

# Para el Metrobus
MB = MB.drop('linea', axis=1)

# Para el RTP
columnasR = ['tipo_pago', 'servicio']
RTP = RTP.drop(columnasR, axis=1)

In [35]:
RTP.head(1)

,fecha,mes,anio,afluencia
0,2022-01-01,Enero,2022.0,0.0


In [36]:
Metro.head(1)

,fecha,anio,mes,afluencia
0,2010-01-01,2010,Enero,20227


In [37]:
MB.head(1)

,fecha,anio,mes,afluencia
0,2005-07-26,2005,Julio,3032667.0


Cambiamos de lugar las columnas del df del RTP

In [38]:
nuevo_orden = ['fecha', 'anio', 'mes', 'afluencia']
RTP = RTP[nuevo_orden]
RTP.head(1)

,fecha,anio,mes,afluencia
0,2022-01-01,2022.0,Enero,0.0


### **Concatenamos los df**

In [39]:
df = pd.concat([RTP, MB, Metro])
df.head()

,fecha,anio,mes,afluencia
0,2022-01-01,2022.0,Enero,0.0
1,2022-01-01,2022.0,Enero,0.0
2,2022-01-01,2022.0,Enero,0.0
3,2022-01-01,2022.0,Enero,2702.0
4,2022-01-01,2022.0,Enero,0.0


# **Usamos merge**

In [44]:
# Seleccionamos solo las columnas necesarias y renombramos afluencia
RTP_df = RTP[['fecha', 'anio', 'mes', 'afluencia']].rename(columns={'afluencia': 'afluencia_RTP'})
MB_df = MB[['fecha', 'anio', 'mes', 'afluencia']].rename(columns={'afluencia': 'afluencia_MB'})
CableBus_df = CableBus[['fecha', 'anio', 'mes', 'afluencia']].rename(columns={'afluencia': 'afluencia_CableBus'})

# Realizamos los merges
merge_1 = pd.merge(RTP_df, MB_df, on=['fecha' ,'anio', 'mes'], how='outer')
df_final = pd.merge(merge_1, CableBus_df, on=['fecha', 'anio', 'mes'], how='outer')

df_final


,fecha,anio,mes,afluencia_RTP,afluencia_MB,afluencia_CableBus
0,2005-07-26,2005.0,Julio,NaN,3032667.0,NaN
1,2005-07-26,2005.0,Julio,NaN,NaN,NaN
2,2005-07-26,2005.0,Julio,NaN,NaN,NaN
3,2005-07-26,2005.0,Julio,NaN,NaN,NaN
4,2005-07-26,2005.0,Julio,NaN,NaN,NaN
...,...,...,...,...,...,...
902284,NaN,NaN,NaN,NaN,NaN,NaN
902285,NaN,NaN,NaN,NaN,NaN,NaN
902286,NaN,NaN,NaN,NaN,NaN,NaN
902287,NaN,NaN,NaN,NaN,NaN,NaN


# **Usamos GroupBy**

In [42]:
# Unificamos las columnas comunes
RTP_df = RTP[['fecha', 'anio', 'mes', 'afluencia']]
MB_df = MB[['fecha', 'anio', 'mes', 'afluencia']]
CableBus_df = CableBus[['fecha', 'anio', 'mes', 'afluencia']]

# Concatenamos los DataFrames
df_unificado = pd.concat([RTP_df, MB_df, CableBus_df], ignore_index=True)

# Agrupamos por año y mes y sumamos la afluencia
df_agrupado = df_unificado.groupby(['fecha', 'anio', 'mes'], as_index=False).sum()

df_agrupado


,fecha,anio,mes,afluencia
0,2005-07-26,2005.0,Julio,3032667.0
1,2005-07-27,2005.0,Julio,230465.0
2,2005-07-28,2005.0,Julio,237296.0
3,2005-07-29,2005.0,Julio,258356.0
4,2005-07-30,2005.0,Julio,159470.0
...,...,...,...,...
7214,2025-04-26,2025.0,Abril,1441187.0
7215,2025-04-27,2025.0,Abril,1044941.0
7216,2025-04-28,2025.0,Abril,1932242.0
7217,2025-04-29,2025.0,Abril,1962249.0
